# NFM Policy Evaluation: Upper Calder Valley

> These are Go notebooks: In order to use the GoNB Jupyter Kernel, please install GoNB from here: https://github.com/janpfeifer/gonb

Note also that for local package development, you can put: `!*go mod edit -replace "github.com/umbralcalc/floodrisk=/path/to/floodrisk"` at the top of any cell.

This notebook runs the full policy evaluation pipeline:
1. Calibrate the rainfall-runoff model from 16 years of observed EA data
2. Fit a stochastic rainfall generator (Markov chain + Gamma distribution)
3. Evaluate 4 NFM portfolios across 5 UKCP18 climate scenarios
4. Visualise peak flow distributions and intervention effectiveness

To run this notebook, you need:
- [GoNB](https://github.com/janpfeifer/gonb) Jupyter kernel installed
- Data downloaded via `go run ./cmd/ingest/` (from the repo root)

## 1. Calibrate, fit rainfall generator, and run policy evaluation

This cell runs the full computation pipeline and prints the results table.

In [ ]:
import (
	"fmt"
	"math/rand/v2"

	"github.com/umbralcalc/floodrisk/pkg/catchment"
	"github.com/umbralcalc/floodrisk/pkg/hydrology"
)

%%

dataDir := "../dat"
rainfallSeries, _ := hydrology.LoadAllRainfallSeries(dataDir)
avgRainfall := hydrology.AverageCatchmentRainfall(rainfallSeries)
ellandFlow, _ := hydrology.LoadTimeSeries(dataDir+"/flow/elland_daily_flow.csv", "Elland")
rainAligned, flowAligned, _, nDays, _ := hydrology.AlignDaily(avgRainfall, ellandFlow)
fmt.Printf("Aligned %d days of observed rainfall and flow data\n\n", nDays)

rainfallData := hydrology.ToStorageData(rainAligned)
rng := rand.New(rand.NewPCG(42, 99))
calResult := catchment.Calibrate(
	rainfallData, flowAligned, catchment.DefaultBounds(), 5000, 30, rng,
)
fmt.Printf("=== Calibration ===\n")
fmt.Printf("NSE: %.4f\n", calResult.NSE)
fmt.Printf("  Field capacity:  %.1f mm\n", calResult.Params["field_capacity"][0])
fmt.Printf("  Drainage rate:   %.4f\n", calResult.Params["drainage_rate"][0])
fmt.Printf("  ET rate:         %.2f mm/day\n", calResult.Params["et_rate"][0])
fmt.Printf("  Runoff shape:    %.2f\n", calResult.Params["runoff_shape"][0])
fmt.Printf("  Fast recession:  %.4f\n", calResult.Params["fast_recession_rate"][0])
fmt.Printf("  Slow recession:  %.4f\n", calResult.Params["slow_recession_rate"][0])
fmt.Printf("  Catchment area:  %.0f km²\n\n", calResult.Params["catchment_area_km2"][0])

wetThreshold := 0.1
shape, scale := catchment.FitGammaParams(rainAligned, wetThreshold)
p01, p11 := catchment.FitWetDryTransitions(rainAligned, wetThreshold)
fmt.Printf("=== Rainfall Generator ===\n")
fmt.Printf("Gamma: shape=%.2f  scale=%.2f mm\n", shape, scale)
fmt.Printf("Markov: P(wet|dry)=%.2f  P(wet|wet)=%.2f\n\n", p01, p11)

rainfallParams := catchment.RainfallParams{
	WetDayShape: shape, WetDayScale: scale,
	PWetGivenDry: p01, PWetGivenWet: p11,
	RainfallMultiplier: 1.0, WetThreshold: wetThreshold,
}

subs := hydrology.UpperCalderSubCatchments()
var subNames []string
for _, sc := range subs {
	if sc.AreaKm2 > 0 {
		subNames = append(subNames, sc.Name)
	}
}
routingCoeffs := make([]float64, len(subNames))
for i := range routingCoeffs {
	routingCoeffs[i] = 0.8
}

cfg := catchment.PolicyEvaluationConfig{
	RunoffParams: calResult.Params, RainfallParams: rainfallParams,
	RoutingCoeffs: routingCoeffs, SubCatchments: subNames,
	NSteps: 3650, NMembers: 50, SpinUp: 30, BaseSeed: 42,
	Priors: catchment.DefaultInterventionPriors(),
}

results := catchment.EvaluatePolicy(catchment.CandidatePortfolios(), catchment.StandardClimateScenarios(), cfg)

fmt.Printf("=== Policy Evaluation Results (%d combinations) ===\n", len(results))
fmt.Printf("%-25s %-15s %10s %10s %10s\n", "Portfolio", "Scenario", "MeanPeak", "P95Peak", "MaxPeak")
fmt.Println("--------------------------------------------------------------------------------")
for _, r := range results {
	fmt.Printf("%-25s %-15s %10.2f %10.2f %10.2f\n",
		r.PortfolioName, r.ScenarioName,
		r.Summary.MeanPeakFlow, r.Summary.P95PeakFlow, r.Summary.MaxPeakFlow)
}

baselinePeaks := make(map[string]float64)
for _, r := range results {
	if r.PortfolioName == "no_intervention" {
		baselinePeaks[r.ScenarioName] = r.Summary.MeanPeakFlow
	}
}
fmt.Printf("\n=== Peak Flow Reduction vs No Intervention ===\n")
fmt.Printf("%-25s %15s %10s\n", "Portfolio", "Scenario", "Reduction")
fmt.Println("----------------------------------------------------")
for _, r := range results {
	if r.PortfolioName == "no_intervention" { continue }
	baseline := baselinePeaks[r.ScenarioName]
	pct := (baseline - r.Summary.MeanPeakFlow) / baseline * 100
	fmt.Printf("%-25s %15s %9.1f%%\n", r.PortfolioName, r.ScenarioName, pct)
}

## 2. Mean peak flow by portfolio and climate scenario

Each portfolio is a separate series; the x-axis is the UKCP18 rainfall multiplier.

In [ ]:
import (
	"math/rand/v2"
	"strconv"

	"github.com/go-echarts/go-echarts/v2/charts"
	"github.com/go-echarts/go-echarts/v2/opts"
	"github.com/go-gota/gota/dataframe"
	"github.com/go-gota/gota/series"
	gonb_echarts "github.com/janpfeifer/gonb-echarts"
	"github.com/umbralcalc/floodrisk/pkg/catchment"
	"github.com/umbralcalc/floodrisk/pkg/hydrology"
	"github.com/umbralcalc/stochadex/pkg/analysis"
)

%%

dataDir := "../dat"
rs, _ := hydrology.LoadAllRainfallSeries(dataDir)
avg := hydrology.AverageCatchmentRainfall(rs)
ef, _ := hydrology.LoadTimeSeries(dataDir+"/flow/elland_daily_flow.csv", "Elland")
ra, fa, _, _, _ := hydrology.AlignDaily(avg, ef)
rd := hydrology.ToStorageData(ra)
rng := rand.New(rand.NewPCG(42, 99))
cal := catchment.Calibrate(rd, fa, catchment.DefaultBounds(), 5000, 30, rng)
sh, sc := catchment.FitGammaParams(ra, 0.1)
p01, p11 := catchment.FitWetDryTransitions(ra, 0.1)
rp := catchment.RainfallParams{WetDayShape: sh, WetDayScale: sc, PWetGivenDry: p01, PWetGivenWet: p11, RainfallMultiplier: 1.0, WetThreshold: 0.1}
subs := hydrology.UpperCalderSubCatchments()
var sn []string
for _, s := range subs { if s.AreaKm2 > 0 { sn = append(sn, s.Name) } }
rc := make([]float64, len(sn))
for i := range rc { rc[i] = 0.8 }
scenarios := catchment.StandardClimateScenarios()
cfg := catchment.PolicyEvaluationConfig{RunoffParams: cal.Params, RainfallParams: rp, RoutingCoeffs: rc, SubCatchments: sn, NSteps: 3650, NMembers: 50, SpinUp: 30, BaseSeed: 42, Priors: catchment.DefaultInterventionPriors()}
results := catchment.EvaluatePolicy(catchment.CandidatePortfolios(), scenarios, cfg)
sm := make(map[string]float64)
for _, s := range scenarios { sm[s.Name] = s.RainfallMultiplier }

var pCol, mCol, mpCol []string
for _, r := range results {
	pCol = append(pCol, r.PortfolioName)
	mCol = append(mCol, strconv.FormatFloat(sm[r.ScenarioName], 'f', 2, 64))
	mpCol = append(mpCol, strconv.FormatFloat(r.Summary.MeanPeakFlow, 'f', 2, 64))
}
df := dataframe.New(
	series.New(mCol, series.Float, "rainfall_multiplier"),
	series.New(mpCol, series.Float, "mean_peak_flow"),
	series.New(pCol, series.String, "portfolio"),
)
scatter := analysis.NewScatterPlotFromDataFrame(&df, "rainfall_multiplier", "mean_peak_flow", "portfolio")
scatter.SetGlobalOptions(
	charts.WithTitleOpts(opts.Title{Title: "Mean Peak Flow by Portfolio and Climate Scenario", Bottom: "1%"}),
	charts.WithXAxisOpts(opts.XAxis{Name: "Rainfall Multiplier"}),
	charts.WithYAxisOpts(opts.YAxis{Name: "Mean Peak Flow (m³/s)", Min: 0}),
)
gonb_echarts.Display(scatter, "width: 1024px; height: 500px; background: white;")

## 3. P95 peak flow by portfolio and climate scenario

In [ ]:
import (
	"math/rand/v2"
	"strconv"

	"github.com/go-echarts/go-echarts/v2/charts"
	"github.com/go-echarts/go-echarts/v2/opts"
	"github.com/go-gota/gota/dataframe"
	"github.com/go-gota/gota/series"
	gonb_echarts "github.com/janpfeifer/gonb-echarts"
	"github.com/umbralcalc/floodrisk/pkg/catchment"
	"github.com/umbralcalc/floodrisk/pkg/hydrology"
	"github.com/umbralcalc/stochadex/pkg/analysis"
)

%%

dataDir := "../dat"
rs, _ := hydrology.LoadAllRainfallSeries(dataDir)
avg := hydrology.AverageCatchmentRainfall(rs)
ef, _ := hydrology.LoadTimeSeries(dataDir+"/flow/elland_daily_flow.csv", "Elland")
ra, fa, _, _, _ := hydrology.AlignDaily(avg, ef)
rd := hydrology.ToStorageData(ra)
rng := rand.New(rand.NewPCG(42, 99))
cal := catchment.Calibrate(rd, fa, catchment.DefaultBounds(), 5000, 30, rng)
sh, sc := catchment.FitGammaParams(ra, 0.1)
p01, p11 := catchment.FitWetDryTransitions(ra, 0.1)
rp := catchment.RainfallParams{WetDayShape: sh, WetDayScale: sc, PWetGivenDry: p01, PWetGivenWet: p11, RainfallMultiplier: 1.0, WetThreshold: 0.1}
subs := hydrology.UpperCalderSubCatchments()
var sn []string
for _, s := range subs { if s.AreaKm2 > 0 { sn = append(sn, s.Name) } }
rc := make([]float64, len(sn))
for i := range rc { rc[i] = 0.8 }
scenarios := catchment.StandardClimateScenarios()
cfg := catchment.PolicyEvaluationConfig{RunoffParams: cal.Params, RainfallParams: rp, RoutingCoeffs: rc, SubCatchments: sn, NSteps: 3650, NMembers: 50, SpinUp: 30, BaseSeed: 42, Priors: catchment.DefaultInterventionPriors()}
results := catchment.EvaluatePolicy(catchment.CandidatePortfolios(), scenarios, cfg)
sm := make(map[string]float64)
for _, s := range scenarios { sm[s.Name] = s.RainfallMultiplier }

var pCol, mCol, p95Col []string
for _, r := range results {
	pCol = append(pCol, r.PortfolioName)
	mCol = append(mCol, strconv.FormatFloat(sm[r.ScenarioName], 'f', 2, 64))
	p95Col = append(p95Col, strconv.FormatFloat(r.Summary.P95PeakFlow, 'f', 2, 64))
}
df := dataframe.New(
	series.New(mCol, series.Float, "rainfall_multiplier"),
	series.New(p95Col, series.Float, "p95_peak_flow"),
	series.New(pCol, series.String, "portfolio"),
)
scatter := analysis.NewScatterPlotFromDataFrame(&df, "rainfall_multiplier", "p95_peak_flow", "portfolio")
scatter.SetGlobalOptions(
	charts.WithTitleOpts(opts.Title{Title: "P95 Peak Flow by Portfolio and Climate Scenario", Bottom: "1%"}),
	charts.WithXAxisOpts(opts.XAxis{Name: "Rainfall Multiplier"}),
	charts.WithYAxisOpts(opts.YAxis{Name: "P95 Peak Flow (m³/s)", Min: 0}),
)
gonb_echarts.Display(scatter, "width: 1024px; height: 500px; background: white;")

## 4. Peak flow reduction vs no intervention (%)

In [ ]:
import (
	"math/rand/v2"
	"strconv"

	"github.com/go-echarts/go-echarts/v2/charts"
	"github.com/go-echarts/go-echarts/v2/opts"
	"github.com/go-gota/gota/dataframe"
	"github.com/go-gota/gota/series"
	gonb_echarts "github.com/janpfeifer/gonb-echarts"
	"github.com/umbralcalc/floodrisk/pkg/catchment"
	"github.com/umbralcalc/floodrisk/pkg/hydrology"
	"github.com/umbralcalc/stochadex/pkg/analysis"
)

%%

dataDir := "../dat"
rs, _ := hydrology.LoadAllRainfallSeries(dataDir)
avg := hydrology.AverageCatchmentRainfall(rs)
ef, _ := hydrology.LoadTimeSeries(dataDir+"/flow/elland_daily_flow.csv", "Elland")
ra, fa, _, _, _ := hydrology.AlignDaily(avg, ef)
rd := hydrology.ToStorageData(ra)
rng := rand.New(rand.NewPCG(42, 99))
cal := catchment.Calibrate(rd, fa, catchment.DefaultBounds(), 5000, 30, rng)
sh, sc := catchment.FitGammaParams(ra, 0.1)
p01, p11 := catchment.FitWetDryTransitions(ra, 0.1)
rp := catchment.RainfallParams{WetDayShape: sh, WetDayScale: sc, PWetGivenDry: p01, PWetGivenWet: p11, RainfallMultiplier: 1.0, WetThreshold: 0.1}
subs := hydrology.UpperCalderSubCatchments()
var sn []string
for _, s := range subs { if s.AreaKm2 > 0 { sn = append(sn, s.Name) } }
rc := make([]float64, len(sn))
for i := range rc { rc[i] = 0.8 }
scenarios := catchment.StandardClimateScenarios()
cfg := catchment.PolicyEvaluationConfig{RunoffParams: cal.Params, RainfallParams: rp, RoutingCoeffs: rc, SubCatchments: sn, NSteps: 3650, NMembers: 50, SpinUp: 30, BaseSeed: 42, Priors: catchment.DefaultInterventionPriors()}
results := catchment.EvaluatePolicy(catchment.CandidatePortfolios(), scenarios, cfg)
sm := make(map[string]float64)
for _, s := range scenarios { sm[s.Name] = s.RainfallMultiplier }

baselinePeaks := make(map[string]float64)
for _, r := range results {
	if r.PortfolioName == "no_intervention" {
		baselinePeaks[r.ScenarioName] = r.Summary.MeanPeakFlow
	}
}
var pCol, mCol, redCol []string
for _, r := range results {
	if r.PortfolioName == "no_intervention" { continue }
	baseline := baselinePeaks[r.ScenarioName]
	pct := (baseline - r.Summary.MeanPeakFlow) / baseline * 100
	pCol = append(pCol, r.PortfolioName)
	mCol = append(mCol, strconv.FormatFloat(sm[r.ScenarioName], 'f', 2, 64))
	redCol = append(redCol, strconv.FormatFloat(pct, 'f', 1, 64))
}
df := dataframe.New(
	series.New(mCol, series.Float, "rainfall_multiplier"),
	series.New(redCol, series.Float, "peak_flow_reduction_pct"),
	series.New(pCol, series.String, "portfolio"),
)
scatter := analysis.NewScatterPlotFromDataFrame(&df, "rainfall_multiplier", "peak_flow_reduction_pct", "portfolio")
scatter.SetGlobalOptions(
	charts.WithTitleOpts(opts.Title{Title: "Peak Flow Reduction vs No Intervention", Bottom: "1%"}),
	charts.WithXAxisOpts(opts.XAxis{Name: "Rainfall Multiplier"}),
	charts.WithYAxisOpts(opts.YAxis{Name: "Reduction (%)", Min: 0}),
)
gonb_echarts.Display(scatter, "width: 1024px; height: 500px; background: white;")